# Data Analysis Toolkit

**Student:** Christin Renni Philip  
**Student ID:** 100008781  
**Course:** Tools and Methods of Data Analysis

In [ ]:
# ==========================================================
# SECTION 1: IMPORT DATASET
# Purpose: Load CSV dataset into the toolkit.
# ==========================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

print("Toolkit libraries loaded successfully.")


## 1. Import Dataset

In [ ]:
# ==========================================================
# SECTION 2: DATA EXPLORATION
# Purpose: Explore rows, columns, and data types.
# ==========================================================
try:
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
except:
    filename = input("Enter CSV file path: ")

df = pd.read_csv(filename, encoding="latin1")
df.head()


## 2. Data Exploration

In [ ]:
# ==========================================================
# SECTION 3: MISSING VALUE ANALYSIS
# Purpose: Identify missing values and duplicate records.
# ==========================================================
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Columns list:", df.columns.tolist())
display(df.dtypes)
df.info()


## 3. Missing-Value Analysis and Duplicate Check

In [ ]:
# ==========================================================
# SECTION 4: DATA CLEANING AND PREPROCESSING
# Purpose: Handle missing values and duplicates.
# ==========================================================
missing_table = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
})
display(missing_table)
print("Duplicate rows:", df.duplicated().sum())


## 4. Data Cleaning and Preprocessing

In [ ]:
# ==========================================================
# SECTION 5: NUMERIC COLUMN SELECTION
# Purpose: Select a variable for analysis.
# ==========================================================
df_clean = df.copy().drop_duplicates()

for col in df_clean.select_dtypes(include=np.number).columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in df_clean.select_dtypes(exclude=np.number).columns:
    if not df_clean[col].mode().empty:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Cleaning completed.")
print("Remaining missing values:", df_clean.isnull().sum().sum())
df_clean.head()


## 5. Select Numeric Column

In [ ]:
# ==========================================================
# SECTION 6: DESCRIPTIVE STATISTICS
# Purpose: Calculate summary measures.
# ==========================================================
numeric_columns = df_clean.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:", numeric_columns)
column_name = input("Enter numeric column name: ")
data = df_clean[column_name].dropna()


## 6. Descriptive Statistics

In [ ]:
# ==========================================================
# SECTION 7: DATA VISUALIZATION
# Purpose: Create histograms and boxplots.
# ==========================================================
summary_table = pd.DataFrame({
    "Statistic": ["Count", "Mean", "Median", "Mode", "Standard Deviation", "Variance", "Minimum", "Maximum", "Q1", "Q3", "Skewness", "Kurtosis"],
    "Value": [data.count(), data.mean(), data.median(), data.mode()[0], data.std(), data.var(), data.min(), data.max(), data.quantile(0.25), data.quantile(0.75), data.skew(), data.kurtosis()]
})
display(summary_table)


## 7. Data Visualization

In [ ]:
# ==========================================================
# SECTION 8: PROBABILITY DISTRIBUTION ANALYSIS
# Purpose: Generate PDF and CDF plots.
# ==========================================================
plt.figure(figsize=(8,5))
plt.hist(data, bins=20, edgecolor="black", density=True)
plt.title("Histogram of " + column_name)
plt.xlabel(column_name)
plt.ylabel("Density")
plt.show()

plt.figure(figsize=(8,4))
plt.boxplot(data, vert=False)
plt.title("Boxplot of " + column_name)
plt.xlabel(column_name)
plt.show()


## 8. Probability Distributions: PDF and CDF

In [ ]:
# ==========================================================
# SECTION 9: Q-Q PLOT ANALYSIS
# Purpose: Assess normality visually.
# ==========================================================
mean = data.mean()
std = data.std()
x = np.linspace(data.min(), data.max(), 100)
normal_pdf = stats.norm.pdf(x, mean, std)

plt.figure(figsize=(8,5))
plt.hist(data, bins=20, density=True, alpha=0.6, edgecolor="black", label="Actual Data")
plt.plot(x, normal_pdf, linewidth=2, label="Fitted Normal Distribution")
plt.title("PDF Plot with Fitted Normal Distribution")
plt.xlabel(column_name)
plt.ylabel("Density")
plt.legend()
plt.show()

sorted_data = np.sort(data)
cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
plt.figure(figsize=(8,5))
plt.plot(sorted_data, cdf)
plt.title("CDF Plot of " + column_name)
plt.xlabel(column_name)
plt.ylabel("Cumulative Probability")
plt.show()


## 9. Q-Q Plot

In [ ]:
# ==========================================================
# SECTION 10: NORMALITY TESTING
# Purpose: Run Shapiro-Wilk, KS, and Anderson-Darling tests.
# ==========================================================
plt.figure(figsize=(6,6))
stats.probplot(data, dist="norm", plot=plt)
plt.title("Q-Q Plot")
plt.show()


## 10. Normality Testing

In [ ]:
# ==========================================================
# SECTION 11: CONFIDENCE INTERVAL ESTIMATION
# Purpose: Calculate confidence intervals.
# ==========================================================
alpha = 0.05
shapiro = stats.shapiro(data)
ks = stats.kstest(data, "norm", args=(data.mean(), data.std()))
anderson = stats.anderson(data, dist="norm")

normality_table = pd.DataFrame({
    "Test": ["Shapiro-Wilk", "Kolmogorov-Smirnov", "Anderson-Darling"],
    "Statistic": [shapiro.statistic, ks.statistic, anderson.statistic],
    "p-value / Info": [shapiro.pvalue, ks.pvalue, "Compare statistic with critical value"],
    "Null Hypothesis": ["Data is normally distributed"] * 3,
    "Alternative Hypothesis": ["Data is not normally distributed"] * 3
})
display(normality_table)

print("Shapiro decision:", "Fail to reject H0" if shapiro.pvalue > alpha else "Reject H0")
print("K-S decision:", "Fail to reject H0" if ks.pvalue > alpha else "Reject H0")
print("Anderson critical values:", anderson.critical_values)
print("Anderson significance levels:", anderson.significance_level)


## 11. Confidence Interval

In [ ]:
# ==========================================================
# SECTION 12: HYPOTHESIS TESTING
# Purpose: Perform one-sample t-test.
# ==========================================================
ci = stats.t.interval(0.95, len(data)-1, loc=data.mean(), scale=stats.sem(data))
print("Sample mean:", data.mean())
print("95% Confidence Interval:", ci)


## 12. Hypothesis Testing: One-Sample T-Test

In [ ]:
# ==========================================================
# SECTION 13: CORRELATION ANALYSIS
# Purpose: Create correlation heatmap.
# ==========================================================
hypothesized_mean = float(input("Enter hypothesized mean value: "))

t_result = stats.ttest_1samp(data, hypothesized_mean)

print("H0: Mean =", hypothesized_mean)
print("H1: Mean !=", hypothesized_mean)
print("T-statistic:", t_result.statistic)
print("p-value:", t_result.pvalue)

if t_result.pvalue > alpha:
    print("Decision: Fail to reject H0")
else:
    print("Decision: Reject H0")


## 13. Correlation Heatmap

In [ ]:
# ==========================================================
# SECTION 14: EXPORT CLEANED DATASET
# Purpose: Save cleaned dataset.
# ==========================================================
numeric_df = df_clean.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    corr = numeric_df.corr()
    plt.figure(figsize=(8,6))
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Correlation Heatmap")
    plt.show()
else:
    print("Not enough numeric columns for correlation heatmap.")


## 14. Export Cleaned Dataset

In [ ]:
df_clean.to_csv("cleaned_dataset.csv", index=False)
print("Cleaned dataset exported as cleaned_dataset.csv")


## Final Conclusion

This toolkit is reusable for CSV datasets and includes data exploration, cleaning, descriptive statistics, visualization, probability distribution analysis, confidence intervals, normality testing, and hypothesis testing.